In [1]:
# Load model directly
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
import swanlab
import json
import os
import pandas as pd
from peft import LoraConfig, get_peft_model, PeftModel
from torch.utils.data import Dataset



In [2]:
# 检查 GPU 是否可用
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", torch_dtype=torch.float16)
model = model.to("cuda:0")
print(f"Model device: {next(model.parameters()).device}")
train_data = pd.read_json("training_data.jsonl", lines=True)

Prompt_dict = {"prompt_no_input": """<|im_start|>system\n{instruction}<|im_end|>\n<|im_start|>user\n<|im_end|>\n<|im_start|>assistant\n""",
    "prompt_input": """<|im_start|>system\n{instruction}<|im_end|>\n<|im_start|>user\n{input}<|im_end|>\n<|im_start|>assistant\n"""
    }
print(type(Prompt_dict))

def json_to_dict(inp, out) -> str:
    prompt = Prompt_dict["prompt_input"].format(instruction="请根据现象分析", input=inp, output=out)
    print(prompt)
    return prompt

GPU Available: True
GPU Count: 1
Current GPU: NVIDIA GeForce RTX 4060 Laptop GPU


`torch_dtype` is deprecated! Use `dtype` instead!


Model device: cuda:0
<class 'dict'>


In [3]:
class SFTDataset(Dataset):
    """监督微调数据集"""
    
    def __init__(self, data_df, tokenizer, prompt_dict, test_mode=False):
        # 使用不同的属性名，避免冲突
        self._data = data_df
        self.tokenizer = tokenizer
        self.prompt_dict = prompt_dict
        self.test_mode = test_mode
        
        if test_mode and len(data_df) > 0:
            print("测试模式：只使用第一条数据")
            self._test_data = [data_df.iloc[0]]
        else:
            self._test_data = None
    
    @property
    def data(self):
        """获取数据的property方法"""
        return self._test_data if self.test_mode else self._data
    
    def __len__(self):
        if self.test_mode:
            return 1  # 测试模式下只有一条数据
        return len(self._data)
    
    def __getitem__(self, idx):
        if self.test_mode:
            row = self._test_data[0]
        else:
            row = self._data.iloc[idx]
        
        instruction = row.get("instruction", "")
        input_text = row.get("input", "")
        output_text = row.get("response", "")
        
        # 构建 prompt
        if input_text:
            prompt = self.prompt_dict["prompt_input"].format(
                instruction=instruction, 
                input=input_text
            )
        else:
            prompt = self.prompt_dict["prompt_no_input"].format(
                instruction=instruction
            )
        
        full_text = prompt + output_text
        
        # tokenize
        encodings = self.tokenizer(
            full_text,
            truncation=True,
            max_length=1024,
            padding="max_length",
            return_tensors="pt"
        )
        
        # 获取input_ids
        input_ids = encodings["input_ids"].squeeze()
        
        # 创建labels（初始与input_ids相同）
        labels = input_ids.clone()
        
        # 计算prompt的长度（需要忽略的部分）
        prompt_encoding = self.tokenizer(
            prompt,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )
        prompt_length = len(prompt_encoding["input_ids"][0])
        
        # 将prompt部分的labels设置为-100（忽略loss）
        labels[:prompt_length] = -100
        
        # 打印调试信息（测试模式）
        if self.test_mode and idx == 0:
            print("\n" + "="*50)
            print("📊 测试数据详情：")
            print("="*50)
            print(f"指令: {instruction}")
            print(f"输入: {input_text[:100]}...")  # 只显示前100字符
            print(f"输出: {output_text[:100]}...")
            print(f"Prompt长度: {prompt_length} tokens")
            print(f"总长度: {len(input_ids)} tokens")
            print(f"需要学习的部分: {len(input_ids) - prompt_length} tokens")
            print("="*50 + "\n")
        
        return {
            "input_ids": input_ids,
            "attention_mask": encodings["attention_mask"].squeeze(),
            "labels": labels
        }

In [4]:
if __name__ == "__main__":
    lora_config = LoraConfig(
        r=16,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
    )
    # 创建数据集
    train_dataset = SFTDataset(train_data, tokenizer, Prompt_dict)
    
    model = get_peft_model(model, lora_config)
    print(model)
    model.print_trainable_parameters()
    print(model)
    args = TrainingArguments(
        report_to="none",
        output_dir="outputs",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=50,
        learning_rate=5e-4,
        lr_scheduler_type="constant",
        logging_steps=1,
        save_steps=5,
        remove_unused_columns=False
    )

    model = model.to("cuda:0")
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset
    )

    trainer.train(resume_from_checkpoint="./checkpoint-29000")


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "d:\anaconda\envs\d2l_env\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "d:\anaconda\envs\d2l_env\Lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "d:\anaconda\envs\d2l_env\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "d:\anaconda\envs\d2l_env\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb2 in position 7: invalid start byte


PeftModel(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_features=1536,

	save_steps: 5 (from args) != 50 (from trainer_state.json)


Step,Training Loss


In [5]:
# 加载 LoRA 参数进行推理
model_with_lora = PeftModel.from_pretrained(model, "./checkpoint-29000")
print("LoRA 模型加载成功")

# 合并 LoRA 参数到基础模型中
model_with_lora = model_with_lora.merge_and_unload()
print("LoRA 参数已合并到基础模型")

# 设置为推理模式
model_with_lora.eval()

# 统计模型参数
total_params = sum(p.numel() for p in model_with_lora.parameters())
print(f"模型总参数数: {total_params:,}")
print(f"模型状态: 推理模式（所有参数已冻结）")


d:\anaconda\envs\d2l_env\Lib\site-packages\peft\config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
d:\anaconda\envs\d2l_env\Lib\site-packages\peft\tuners\tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
d:\anaconda\envs\d2l_env\Lib\site-packages\peft\peft_model.py:585: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default

LoRA 模型加载成功
LoRA 参数已合并到基础模型
模型总参数数: 1,543,714,304
模型状态: 推理模式（所有参数已冻结）


In [ ]:
messages = [
    {"role": "user", "content": '''Discovery of quasi-periodic oscillations in the new X-ray pulsar XTE J1858+034,"Discovery of quasi-periodic oscillations in the X-ray pulsar XTE J1858+034
# Discovery of quasi-periodic oscillations in the new X-ray pulsar XTE J1858+034
### B. Paul  
Department of Astronomy and Astrophysics  
Tata Institute of Fundamental Research  
Homi Bhabha Road, Mumbai 400 005, India
We have discovered low frequency quasi-periodic oscillations (QPO) at a frequency of 0.11 Hz in the newly discovered 221 s X-ray pulsar XTE J1858+034 (IAUC 6826, 6828). We have used public archival data of four observations of this source made with the PCA detectors of the RXTE during 1998, February 20 and 24. The QPO feature in the power density spectrum (PDS) is centered at 0.11 ± 0.01 Hz and the rms variability at the QPO frequency is ~6.5%. Apart from the gaussian QPO feature, the PDS, in the frequency range of 0.006-0.6 Hz is power-law type with an index of -0.95. The energy spectrum in 2-60 keV range has been analyzed for one of the observations and it is found to be very hard and has the following components :  
a) black body component of temperature 3.5 ± 0.2 keV,  
b) iron fluroscence line at 6.60 ± 0.07 keV of equivalent width 200 ± 40ev,   
c) power-law component with photon spectral index of 1.56 ± 0.10 and  
d) an ionized absorber.  
Total incident flux in the 1.3 - 100 keV observation band of the PCA detectors, is 9 × 10-10 erg cm-2 s-1. 
This is the fifth X-ray pulsar after Cen X-3, EXO 2030+375, 4U 1626-67 and GRO J1744-28 in which QPOs have been detected. Assuming that the QPOs are produced as a result of some inhomogeneity at the magnetospheric boundary with the disk rotating at the Keplerian frequency (fK), the Keplerian frequency fK is derived to be 0.11 Hz . For a neutron star of mass 1.4 MSun, this indicates that the magnetosphere radius rM is 7.3 × 108 cm. From the X-ray luminosity of this pulsar for an assumed distance of rkpc, the magnetic field of this pulsar is estimated to be about ~1012 rkpc Gauss, where rkpc is the distance of the pulsar expressed in kiloparsec. 
The Keplerian frequency of the disk at the magnetospheric radius is ~ 0.2 Hz in three of these five pulsars in which QPOs are observed. These are Cen X-3, EXO 23030+375 and 4U 1626-67. In the bursting pulsar GRO J1744-28 the magnetic field strength is estimated to be much smaller which is consistent with a smaller magnetosphere with Keplerian frequency of the disk at magnetosphere being 40 Hz. This new pulsar with lowest fK appears to be the one having a largest magnetosphere.",1998-03-19 16:59:00
BeppoSAX Alert -- GB980425,"I received the following notice: 
BeppoSAX GRB MAIL N. 98/9  
^^^^^^^^^^^^^^^^^^^^^^^^^  

GB980425 
The BeppoSAX GRBM was triggered at 21:49:11 UT of April 25 by a GRB (GB980425) The event was detected by one of the WFC (WFC2), with preliminary position:  
RA(2000)=293.65  
Dec(2000)=-52.81  
The error radius at this stage is 10', that includes systematic errors due to a not optimal attitude configuration. 
An update of the position will be released soon. 
A BeppoSAX follow up with the NFI is being planned 
Luigi Piro'''
    }
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model_with_lora.device)

outputs = model_with_lora.generate(**inputs, max_new_tokens=25600, do_sample=True, temperature=0.7)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(f"Qwen (with LoRA): {response}\n")

Qwen (with LoRA): The discovery of quasi-periodic oscillations (QPOs) in the newly identified 221-second X-ray pulsar XTE J1858+034 by B. Paul et al. is significant because it provides insights into the behavior of neutron stars, particularly those with strong magnetic fields. Here's a summary of key points:

**Observation Details:**
- **Source:** XTE J1858+034
- **Type:** 221-second X-ray pulsar
- **Date:** February 20 and 24, 1998
- **Instrument:** PCA detectors of the RXTE

**Key Findings:**

1. **Quasi-Peak Oscillations (QPO):**
   - Discovered at a frequency of 0.11 Hz.
   - RMS variability at the QPO frequency is approximately 6.5%.

2. **Power Spectrum Characteristics:**
   - Power-density spectrum (PDS) shows a Gaussian-like QPO feature at 0.11 Hz.
   - Beyond 0.006-0.6 Hz, the PDS exhibits a power-law behavior with an index of -0.95.

3. **Energy Spectra Analysis:**
   - One of the observations showed an extremely hard energy spectrum between 2-60 keV.
   - Components included